# Creating and Registering a Custom Node

This notebook shows how to implement a custom transform node from scratch and plug it into a TSUT pipeline.

We will build a **Log Transform** node that applies `log1p` (i.e. `log(1 + x)`) to all numerical columns. The steps are:

1. Define the configuration classes (Metadata, HyperParameters, RunningConfig, Config)
2. Implement the node class (fit / transform / get_params / set_params)
3. Register it in the `NODE_REGISTRY`
4. Use it in a pipeline

In [ ]:
import sys

from tsut import NODE_REGISTRY
from tsut.core.common.logging import configure

configure(level="INFO", stream=sys.stdout, fmt="text")

## Step 1 -- Configuration Classes

Every TSUT node needs four configuration pieces:

| Class | Purpose |
|---|---|
| **Metadata** | Node name and description (used by the registry and rendering) |
| **HyperParameters** | Tuneable parameters exposed to hyperparameter search |
| **RunningConfig** | Execution-time settings that stay fixed during tuning |
| **Config** | Bundles the above together with port definitions |

Our Log Transform has one hyperparameter (`base` -- the log base, defaulting to natural log) and no running config.

In [ ]:
from typing import Any

import numpy as np
import pandas as pd
from pydantic import Field

from tsut.core.common.data.data import (
    ArrayLikeEnum,
    DataCategoryEnum,
    DataStructureEnum,
    TabularDataContext,
)
from tsut.core.nodes.node import Port
from tsut.core.nodes.transform.transform import (
    TransformConfig,
    TransformHyperParameters,
    TransformMetadata,
    TransformNode,
    TransformRunningConfig,
)


# -- Metadata --
class LogTransformMetadata(TransformMetadata):
    node_name: str = "LogTransform"
    description: str = "Applies log1p to all numerical columns."
    trainable: bool = False  # Stateless transform -- nothing to fit


# -- HyperParameters (tuneable) --
class LogTransformHyperParameters(TransformHyperParameters):
    base: float = Field(
        default=np.e,
        gt=0,
        description="Logarithm base. Use e for natural log, 10 for log10, etc.",
    )


# -- RunningConfig (empty for this node) --
class LogTransformRunningConfig(TransformRunningConfig):
    pass


# -- Config (bundles everything + ports) --
class LogTransformConfig(
    TransformConfig[LogTransformHyperParameters, LogTransformRunningConfig]
):
    hyperparameters: LogTransformHyperParameters = Field(
        default_factory=LogTransformHyperParameters
    )
    running_config: LogTransformRunningConfig = Field(
        default_factory=LogTransformRunningConfig
    )
    in_ports: dict[str, Port] = Field(
        default={
            "input": Port(
                arr_type=ArrayLikeEnum.PANDAS,
                data_structure=DataStructureEnum.TABULAR,
                data_category=DataCategoryEnum.NUMERICAL,
                data_shape="batch features",
                desc="Numerical feature DataFrame.",
            ),
        }
    )
    out_ports: dict[str, Port] = Field(
        default={
            "output": Port(
                arr_type=ArrayLikeEnum.PANDAS,
                data_structure=DataStructureEnum.TABULAR,
                data_category=DataCategoryEnum.NUMERICAL,
                data_shape="batch features",
                desc="Log-transformed feature DataFrame.",
            ),
        }
    )


print("Configuration classes defined.")

## Step 2 -- The Node Class

The node class inherits from `TransformNode` and must implement four methods:

- **`fit(data)`** -- learn internal state (no-op for stateless transforms)
- **`transform(data)`** -- apply the transformation and return `{port_name: (array, context)}`
- **`get_params()`** / **`set_params(params)`** -- serialise and restore learned state

The generic type parameters are `[InputData, InputContext, OutputData, OutputContext, ParamsType]`.

In [ ]:
type _LogTransformParams = dict[str, Any]


class LogTransformNode(
    TransformNode[
        pd.DataFrame,          # Input data type
        TabularDataContext,     # Input context type
        pd.DataFrame,          # Output data type
        TabularDataContext,     # Output context type
        _LogTransformParams,   # Params type (empty dict for stateless)
    ]
):
    metadata = LogTransformMetadata()

    def __init__(self, *, config: LogTransformConfig) -> None:
        self._config = config
        self._fitted = True  # Stateless -- always "fitted"

    def fit(
        self, data: dict[str, tuple[pd.DataFrame, TabularDataContext]]
    ) -> None:
        """Nothing to learn -- the transform is stateless."""

    def transform(
        self, data: dict[str, tuple[pd.DataFrame, TabularDataContext]]
    ) -> dict[str, tuple[pd.DataFrame, TabularDataContext]]:
        """Apply log1p (in the configured base) to all columns."""
        df, ctx = data["input"]
        base = self._config.hyperparameters.base

        # np.log1p computes log(1+x) in base e; change-of-base if needed
        if base == np.e:
            result = np.log1p(df)
        else:
            result = np.log1p(df) / np.log(base)

        return {"output": (pd.DataFrame(result, columns=df.columns), ctx)}

    def get_params(self) -> _LogTransformParams:
        return {}

    def set_params(self, params: _LogTransformParams) -> None:
        pass


print("LogTransformNode class defined.")

## Step 3 -- Register in the NODE_REGISTRY

Once defined, a node must be registered so that `Pipeline` can instantiate it by name. Registration requires the node class, its config class, and optionally the running config and hyperparameters classes.

In [ ]:
NODE_REGISTRY.register(
    name="LogTransform",
    node_class=LogTransformNode,
    node_config_class=LogTransformConfig,
    running_config_class=LogTransformRunningConfig,
    hyperparameters_class=LogTransformHyperParameters,
)

# Verify it appears in the registry
assert "LogTransform" in NODE_REGISTRY
print("LogTransform registered! Full registry:")
NODE_REGISTRY.list()

## Step 4 -- Use it in a Pipeline

Now `"LogTransform"` can be referenced by name just like any built-in node. We'll build a small pipeline:

`CSVFetcher -> LogTransform -> Sink`

This demonstrates that the custom node integrates seamlessly with the rest of the framework.

In [ ]:
from tsut.core.pipeline.pipeline import Edge, Pipeline, PipelineConfig
from tsut.core.pipeline.runners.smart_runner import SmartRunner

# Configure the CSV source
csv_config = NODE_REGISTRY.get_node_config_class("TabularCSVFetcher")()
csv_config.running_config.csv_path = "../data/fake_batch.csv"
csv_config.running_config.context_path = "../data/fake_batch_context.json"

# Configure our custom LogTransform (use base-10 log as an example)
log_config = LogTransformConfig()
log_config.hyperparameters.base = 10.0

# Build the pipeline
pipe = Pipeline(
    config=PipelineConfig(
        name="Custom Node Demo",
        nodes={
            "source": ("TabularCSVFetcher", csv_config),
            "log_transform": ("LogTransform", log_config),
            "sink": ("Sink", NODE_REGISTRY.get_node_config_class("Sink")()),
        },
        edges=[
            Edge(source="source", target="log_transform", ports_map=[("output", "input")]),
            Edge(source="log_transform", target="sink", ports_map=[("output", "dump")]),
        ],
    )
)

pipe.compile()
pipe.render()

In [ ]:
# Run it
runner = SmartRunner(pipe)
runner.train()
result = runner.infer()

print("Sink output keys:", list(result.keys()))
print("Output shape:", result["dump"].shape)

## Summary

To create a custom node in TSUT:

1. **Define 4 config classes** -- Metadata, HyperParameters, RunningConfig, Config (with port definitions)
2. **Implement the node class** -- inherit from `TransformNode` (or `Model`, `MetricNode`, etc.) and implement the required abstract methods
3. **Register it** -- call `NODE_REGISTRY.register(...)` with the node name, class, and config classes
4. **Use it** -- reference the node by name in any `PipelineConfig`, just like built-in nodes

For nodes that ship with the package (rather than one-off notebook definitions), place them in `src/tsut/components/` with a `_register.py` module and they will be auto-discovered at import time.